# Revue des symboles Yahoo

**Autrice :** Faty Barbosa Monteiro

**Date :** 23 octobre 2026

**Client :** CLIENT001

Ce notebook prépare la décision sur les symboles Yahoo du référentiel qui ne sont pas dans le fichier de cours. Les effectifs sont ceux des fichiers lus.

La feuille `audit/revue_identite.csv` n'est écrite que s'il reste un symbole à décider. Quand chaque symbole configuré est directement validé, aucune feuille n'est créée. Ce notebook ne modifie ni le référentiel, ni les cours, ni le registre ODD.

Le connecteur ne remplace pas le symbole configuré par un symbole proposé. La confirmation se note dans `symbole_yahoo_confirme`. La collecte reprend ensuite ce symbole.


## 1. Comment noter

`decision` prend l'une de ces trois mentions : `confirmer`, `laisser ouvert`, `hors univers francs`. `symbole_yahoo_confirme` reprend le symbole configuré, ou le symbole proposé, seulement après lecture de la page de la bourse ou de la société. `preuve` nomme cette page. `date_revue` est la date de cette lecture.

Une nouvelle exécution conserve les cellules déjà remplies.


In [1]:
import re
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
for candidate in [PROJECT_ROOT, *PROJECT_ROOT.parents]:
    if (candidate / "src" / "config.py").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError("Project root not found from the notebook directory.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.acquisition.connectors.yfinance import DIRECT_VALIDATED_STATUS
from src.config import AUDIT_DIR, TECHNICAL_LOG
from src.governance.source_registry import (
    build_dataset_absolute_path,
    get_dataset_by_output_file,
)
from src.preparation.quality_treatment import CHF_UNIVERSE

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 220)


def registry_output(file_name):
    dataset = get_dataset_by_output_file(file_name)
    return build_dataset_absolute_path(dataset.to_dict())


MASTER_FILE = registry_output("securities_master.csv")
PRICES_FILE = registry_output("market_prices.csv")
UNIVERSE_FILE = CHF_UNIVERSE
REVIEW_FILE = AUDIT_DIR / "revue_identite.csv"
DECISION_COLUMNS = [
    "decision",
    "symbole_yahoo_confirme",
    "preuve",
    "date_revue",
]
KEY_COLUMNS = ["ticker", "exchange"]


## 2. Fichiers lus

Les chemins viennent du registre et de la configuration. Le fichier de cours retient un symbole lorsque son statut est directement validé.


In [2]:
def read_table(path):
    if not path.exists():
        raise FileNotFoundError(path)
    return pd.read_csv(path, dtype=str, keep_default_na=False)


master = read_table(MASTER_FILE)
prices = pd.read_csv(
    PRICES_FILE,
    dtype=str,
    keep_default_na=False,
    usecols=["configured_symbol", "symbol_resolution_status"],
)
universe = read_table(UNIVERSE_FILE)
acquired = set(
    prices.loc[
        prices["symbol_resolution_status"].str.strip().eq(DIRECT_VALIDATED_STATUS),
        "configured_symbol",
    ].str.strip()
)
print(f"Référentiel : {MASTER_FILE}")
print(f"Lignes du référentiel : {len(master)}")
print(f"Symboles directement validés : {len(acquired)}")
print(f"Liste convertible : {UNIVERSE_FILE}")
print(f"Feuille de revue : {REVIEW_FILE}")


Référentiel : C:\Users\MameBARBOSAMONTEIRO\projet01\data\02_processed\securities\securities_master.csv
Lignes du référentiel : 120
Symboles directement validés : 120
Liste convertible : C:\Users\MameBARBOSAMONTEIRO\projet01\data\03_universe\eligible\chf_portfolio_universe.csv
Feuille de revue : C:\Users\MameBARBOSAMONTEIRO\projet01\audit\revue_identite.csv


## 3. Symboles absents du fichier de cours

Le statut et le symbole proposé sont la dernière ligne du journal technique pour ce symbole. Une proposition vide signifie qu'aucun symbole alternatif n'a été retenu.


In [3]:
pattern = re.compile(
    r"Yahoo identity not directly validated: (?P<symbol>.*?); Status=(?P<status>.*?); Candidate=(?P<candidate>.*)$"
)
latest = {}
if TECHNICAL_LOG.exists():
    for line in TECHNICAL_LOG.read_text(encoding="utf-8", errors="replace").splitlines():
        match = pattern.search(line)
        if match:
            latest[match.group("symbol").strip()] = match.groupdict()

rows = []
for _, security in master.iterrows():
    symbol = str(security["yahoo_symbol"]).strip()
    if symbol in acquired:
        continue
    found = latest.get(symbol, {})
    candidate = str(found.get("candidate", "")).strip()
    if candidate.casefold() == "none":
        candidate = ""
    universe_row = universe.loc[
        universe["ticker"].eq(security["Ticker"])
        & universe["exchange"].eq(security["Exchange"])
    ]
    situation = ""
    motif = ""
    clean_rows = ""
    if not universe_row.empty:
        situation = universe_row.iloc[0]["chf_status"]
        motif = universe_row.iloc[0]["exclusion_reason"]
        clean_rows = universe_row.iloc[0]["clean_price_rows"]
    rows.append({
        "ticker": security["Ticker"],
        "nom": security["Name"],
        "exchange": security["Exchange"],
        "devise_cotation": security["listing_currency"],
        "situation_chf": situation,
        "motif_chf": motif,
        "cours_propres": clean_rows,
        "symbole_yahoo": symbol,
        "statut_yahoo": found.get("status", ""),
        "symbole_propose": candidate,
    })
review = pd.DataFrame(
    rows,
    columns=[
        "ticker",
        "nom",
        "exchange",
        "devise_cotation",
        "situation_chf",
        "motif_chf",
        "cours_propres",
        "symbole_yahoo",
        "statut_yahoo",
        "symbole_propose",
    ],
)
print(f"Symboles absents du fichier de cours : {len(review)}")
review


Symboles absents du fichier de cours : 0


,ticker,nom,exchange,devise_cotation,situation_chf,motif_chf,cours_propres,symbole_yahoo,statut_yahoo,symbole_propose


## 4. Feuille de revue

`symbole_yahoo_confirme` n'est accepté que s'il reprend le symbole configuré ou le symbole proposé.


In [4]:
if REVIEW_FILE.exists() and not review.empty:
    previous = read_table(REVIEW_FILE)
    keep = [column for column in DECISION_COLUMNS if column in previous.columns]
    if "ecart" in previous.columns:
        previous = previous.loc[previous["ecart"].eq("Symbole Yahoo")]
    if set(KEY_COLUMNS).issubset(previous.columns):
        previous = previous.drop_duplicates(KEY_COLUMNS, keep="last")
        review = review.merge(previous[KEY_COLUMNS + keep], on=KEY_COLUMNS, how="left")
for column in DECISION_COLUMNS:
    if column not in review.columns:
        review[column] = ""
    review[column] = review[column].fillna("")

warnings = []
for _, row in review.iterrows():
    confirmed = row["symbole_yahoo_confirme"].strip()
    allowed = {row["symbole_yahoo"].strip(), row["symbole_propose"].strip()} - {""}
    if confirmed and confirmed not in allowed:
        warnings.append(
            f"{row['ticker']} : {confirmed} n'est ni le symbole configuré ni le symbole proposé."
        )
if warnings:
    print("\n".join(warnings))
else:
    print("Aucune confirmation hors des symboles déjà vus.")

if review.empty:
    if REVIEW_FILE.exists():
        REVIEW_FILE.unlink()
    print("Chaque symbole configuré est directement validé. Aucune feuille de revue n'est écrite.")
else:
    REVIEW_FILE.parent.mkdir(parents=True, exist_ok=True)
    review.to_csv(REVIEW_FILE, index=False, encoding="utf-8-sig")
    print(f"Feuille enregistrée : {REVIEW_FILE}")
review


Aucune confirmation hors des symboles déjà vus.
Chaque symbole configuré est directement validé. Aucune feuille de revue n'est écrite.


,ticker,nom,exchange,devise_cotation,situation_chf,motif_chf,cours_propres,symbole_yahoo,statut_yahoo,symbole_propose,decision,symbole_yahoo_confirme,preuve,date_revue
